In [ ]:
"""
===============================================================================
Script Name: 01_gpu_ode_calibration.py
Description: Loads chunked tensor segments and trains a Physics-Informed 
             ODE to calibrate thermal parameters. Implements a dynamic
             state-dependent cooling proxy h(T) to simulate fan curves.
             Includes interactive GPU selection, auto-resume, and CSV logging.
===============================================================================
"""

import os
import sys
import time
import warnings
import json
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from tqdm import tqdm
from datetime import timedelta
import gc

# --- 1. CONFIGURATION ---
warnings.filterwarnings('ignore')

# Hyperparameters
BATCH_SIZE = 25000
EPOCHS = 50
LEARNING_RATE = 0.01
DT = 0.11  # Timestep in seconds

# ==========================================
# 2. GPU SELECTION & INITIALIZATION
# ==========================================
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    if not torch.cuda.is_available():
        print("[WARNING] No CUDA GPUs detected. Falling back to CPU.")
        return None

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

# --- 3. TERMINAL LOGGING UTILITY ---
class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "a", encoding="utf-8") # Append mode for resuming

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# --- 4. PHYSICS MODEL (THE ODE) ---
class ThermalODEModel(nn.Module):
    def __init__(self):
        super(ThermalODEModel, self).__init__()
        
        # 1. Define Priors (Starting Guesses)
        prior_C         = 141.735
        prior_k01       = 0.0780
        prior_k10       = 0.0281
        prior_q         = 0.0
        
        # Dynamic Fan Curve Priors
        prior_h_base    = 2.0   # Baseline passive cooling
        prior_h_active  = 8.0   # Max additional cooling from fans
        prior_T_thresh  = 65.0  # Temp where fans start to aggressively ramp up
        prior_beta      = 0.5   # Steepness of the fan curve ramp
        
        # 2. Define Physical Bounds [lower, upper]
        self.bounds = {
            'C':          (50.0, 300.0),
            'k01':        (0.0, 0.5),
            'k10':        (0.0, 0.5),
            'q0':         (-10.0, 10.0),
            'q1':         (-10.0, 10.0),
            'h_base_0':   (0.5, 10.0),
            'h_base_1':   (0.5, 10.0),
            'h_active_0': (0.0, 30.0),
            'h_active_1': (0.0, 30.0),
            'T_thresh_0': (40.0, 85.0),
            'T_thresh_1': (40.0, 85.0),
            'beta_0':     (0.05, 2.0),
            'beta_1':     (0.05, 2.0)
        }
        
        # 3. Initialize Raw Parameters (Inverse Sigmoid)
        def inv_sigmoid(val, low, high):
            norm = (val - low) / (high - low)
            norm = max(min(norm, 0.999), 0.001)
            return torch.log(torch.tensor(norm / (1.0 - norm)))

        # General Params
        self.raw_C          = nn.Parameter(inv_sigmoid(prior_C, *self.bounds['C']))
        self.raw_k01        = nn.Parameter(inv_sigmoid(prior_k01, *self.bounds['k01']))
        self.raw_k10        = nn.Parameter(inv_sigmoid(prior_k10, *self.bounds['k10']))
        self.raw_q0         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q0']))
        self.raw_q1         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q1']))
        
        # GPU 0 Fan Curve Params
        self.raw_h_base_0   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_0']))
        self.raw_h_active_0 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_0']))
        self.raw_T_thresh_0 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_0']))
        self.raw_beta_0     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_0']))
        
        # GPU 1 Fan Curve Params
        self.raw_h_base_1   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_1']))
        self.raw_h_active_1 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_1']))
        self.raw_T_thresh_1 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_1']))
        self.raw_beta_1     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_1']))

    def get_physical_params(self):
        """Applies sigmoid to raw parameters to bind them within physical reality."""
        p = {}
        for name, bound in self.bounds.items():
            raw_val = getattr(self, f"raw_{name}")
            low, high = bound
            p[name] = low + torch.sigmoid(raw_val) * (high - low)
        return p

    def forward(self, P0, P1, T0_init, T1_init, T_amb):
        """Simulates the thermal trajectories using Forward Euler integration."""
        params = self.get_physical_params()
        
        batch_size, seq_len = P0.shape
        
        # Pre-allocate to prevent torch.stack memory fragmentation
        T0_preds = torch.empty((batch_size, seq_len), device=P0.device)
        T1_preds = torch.empty((batch_size, seq_len), device=P1.device)
        
        T0_curr = T0_init
        T1_curr = T1_init
        T0_preds[:, 0] = T0_curr
        T1_preds[:, 0] = T1_curr
        
        inv_C = 1.0 / params['C']
        
        # Euler Integration Loop
        for t in range(seq_len - 1):
            # 1. Calculate Dynamic Cooling Rate h(T) for current timestep
            h0_curr = params['h_base_0'] + params['h_active_0'] * torch.sigmoid(params['beta_0'] * (T0_curr - params['T_thresh_0']))
            h1_curr = params['h_base_1'] + params['h_active_1'] * torch.sigmoid(params['beta_1'] * (T1_curr - params['T_thresh_1']))
            
            # 2. Calculate differentials
            dT0 = (P0[:, t] + params['k01'] * P1[:, t] - h0_curr * T0_curr + h0_curr * T_amb + params['q0']) * inv_C
            dT1 = (P1[:, t] + params['k10'] * P0[:, t] - h1_curr * T1_curr + h1_curr * T_amb + params['q1']) * inv_C
            
            # 3. Euler Step
            T0_curr = T0_curr + DT * dT0
            T1_curr = T1_curr + DT * dT1
            
            T0_preds[:, t+1] = T0_curr
            T1_preds[:, t+1] = T1_curr
            
        return T0_preds, T1_preds


# --- 5. MAIN EXECUTION ---
def main():
    global script_name
    try:
        # Works when running as a normal .py script
        script_path = Path(__file__).resolve()
        script_name = script_path.stem
        project_root = script_path.parent.parent.parent
    except NameError:
        # Fallback for Jupyter Notebooks
        script_name = "01_gpu_ode_calibration"
        current_dir = Path.cwd() 
        project_root = current_dir.parent.parent
    
    # --- Device Setup ---
    gpu_id = select_gpu()
    if gpu_id is not None:
        torch.cuda.set_device(gpu_id)
        DEVICE = torch.device(f"cuda:{gpu_id}")
    else:
        DEVICE = torch.device("cpu")

    # --- Versioning & Path Initialization ---
    user_prefix = input("\nEnter version prefix (e.g., v1_base) [Press Enter for default 'v1']: ").strip()
    VERSION_PREFIX = user_prefix if user_prefix else "v1"

    # Input Data Directory
    data_dir = project_root / "Implementation" / "data" / "mit-supercloud-dataset" / "gpu" / "dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_tensors"
    if not data_dir.exists():
        print(f"[ERROR] Cannot find tensor directory: {data_dir}")
        return

    prefix = "/home/sanke24839/"
    # Output Save Directory
    SAVE_DIR = project_root / "Implementation" / "models" / VERSION_PREFIX
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    
    log_path = SAVE_DIR / f"{script_name}_terminal_output.txt"
    sys.stdout = DualLogger(log_path)
    
    start_time = time.perf_counter()
    print("=" * 70)
    print(f"--- STARTING: {script_name.upper()} ---")
    print(f"[*] Version Prefix: {VERSION_PREFIX}")
    print(f"[*] Device: {DEVICE}")
    print(f"[*] Saving to: {str(SAVE_DIR).replace(prefix, '')}")
    print("=" * 70)

    # --- Background GPU Time-Series Logging ---
    gpu_logger_process = None
    if gpu_id is not None:
        gpu_csv_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu{gpu_id}_timeseries.csv"
        try:
            # Queries specific metrics for the chosen GPU (-i), loops every 5 seconds (-l 5), writes to file (-f)
            smi_cmd = [
                "nvidia-smi",
                "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu,power.draw",
                "--format=csv",
                "-i", str(gpu_id),
                "-l", "5", 
                "-f", str(gpu_csv_path)
            ]
            gpu_logger_process = subprocess.Popen(smi_cmd)
            print(f"[*] Started continuous GPU logging to: {gpu_csv_path.name}")
        except FileNotFoundError:
            print("[WARNING] nvidia-smi not found. Continuous GPU logging disabled.")

    # --- Setup Model, Optimizer, & Criterions ---
    model = ThermalODEModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    mse_criterion = nn.MSELoss(reduction='none') # For optimization
    mae_criterion = nn.L1Loss(reduction='none')  # For interpretable logging

    # --- Auto-Resume Logic ---
    RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
    BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
    CSV_LOG_PATH = SAVE_DIR / "training_log.csv"

    start_epoch = 1
    best_val_loss = float('inf')

    if RESUME_PATH.exists():
        print(f"\n[SYSTEM] Found existing checkpoint at: {RESUME_PATH.name}")
        print(f"[SYSTEM] Resuming training...")
        
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        
        # Restore RNG states
        torch.set_rng_state(checkpoint['torch_rng'].cpu().to(torch.uint8) if isinstance(checkpoint['torch_rng'], torch.Tensor) else torch.ByteTensor(checkpoint['torch_rng']))
        np.random.set_state(checkpoint['numpy_rng'])
        
        if torch.cuda.is_available() and 'cuda_rng' in checkpoint and checkpoint['cuda_rng'] is not None:
            cuda_rng = checkpoint['cuda_rng']
            torch.cuda.set_rng_state(cuda_rng.cpu().to(torch.uint8) if isinstance(cuda_rng, torch.Tensor) else torch.ByteTensor(cuda_rng))
            
        print(f"[SYSTEM] Resumed from Epoch {checkpoint['epoch']}. Best Val RMSE: {best_val_loss:.6f}")
    else:
        print(f"\n[SYSTEM] No checkpoint found. Starting fresh training.")
        # Initialize CSV header if starting fresh
        with open(CSV_LOG_PATH, "w") as f:
            f.write("epoch,train_rmse,val_rmse,train_mae,val_mae,current_lr,C,h_base_0,h_base_1,h_active_0,h_active_1,T_thresh_0,T_thresh_1,beta_0,beta_1,k01,k10,q0,q1,epoch_time_sec\n")

    # Discover chunks
    train_chunks = sorted(list(data_dir.glob("train_segments_part*.pt")))
    val_chunks = sorted(list(data_dir.glob("val_segments_part*.pt")))
    print(f"Found {len(train_chunks)} train chunks and {len(val_chunks)} val chunks.")

    try:
        # --- Training Loop ---
        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_start_time = time.perf_counter()
            print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
            
            # 1. TRAINING PHASE
            model.train()
            train_mse_accum = 0.0
            train_mae_accum = 0.0
            train_steps = 0
            
            for chunk_idx, chunk_file in enumerate(train_chunks):
                chunk_data = torch.load(chunk_file, map_location='cpu', weights_only=False)
                dataset = TensorDataset(
                    chunk_data['P0'], chunk_data['P1'], 
                    chunk_data['T0'], chunk_data['T1'],
                    chunk_data['T_amb'], chunk_data['valid_len']
                )
                dataloader = DataLoader(
                    dataset,
                    batch_size=BATCH_SIZE,
                    shuffle=True,
                    num_workers=4,          # parallel CPU workers prefetch data
                    pin_memory=True,        # faster CPU→GPU transfer
                    persistent_workers=True
                )
                    
                for batch in tqdm(dataloader, desc=f"Train Chunk {chunk_idx+1}/{len(train_chunks)}", leave=False):
                    P0, P1, T0_true, T1_true, T_amb, valid_len = [
                        b.to(DEVICE, non_blocking=True) for b in batch
                    ]
                    
                    optimizer.zero_grad()
                    
                    # Forward pass
                    T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                    T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                    
                    # Create mask for valid timesteps
                    seq_len = P0.shape[1]
                    mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                    
                    # Calculate MSE (for backprop)
                    mse_0 = mse_criterion(T0_pred, T0_true)
                    mse_1 = mse_criterion(T1_pred, T1_true)
                    masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                    
                    # Calculate MAE (for tracking interpretability)
                    with torch.no_grad():
                        mae_0 = mae_criterion(T0_pred.detach(), T0_true)
                        mae_1 = mae_criterion(T1_pred.detach(), T1_true)
                        masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())
    
                    # Backpropagate
                    masked_mse.backward()
                    optimizer.step()
                    
                    train_mse_accum += masked_mse.item()
                    train_mae_accum += masked_mae.item()
                    train_steps += 1
                    
                del chunk_data, dataset, dataloader
    
            avg_train_mse = train_mse_accum / train_steps
            train_rmse = avg_train_mse ** 0.5
            train_mae = train_mae_accum / train_steps
            
            # 2. VALIDATION PHASE
            model.eval()
            val_mse_accum = 0.0
            val_mae_accum = 0.0
            val_steps = 0
            
            with torch.no_grad():
                for chunk_idx, chunk_file in enumerate(val_chunks):
                    chunk_data = torch.load(chunk_file, map_location='cpu', weights_only=False)
                    dataset = TensorDataset(
                        chunk_data['P0'], chunk_data['P1'], 
                        chunk_data['T0'], chunk_data['T1'],
                        chunk_data['T_amb'], chunk_data['valid_len']
                    )
                    dataloader = DataLoader(
                        dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        num_workers=4,          # parallel CPU workers prefetch data
                        pin_memory=True,        # faster CPU→GPU transfer
                        persistent_workers=True
                    )
                    
                    for batch in dataloader:
                        P0, P1, T0_true, T1_true, T_amb, valid_len = [
                            b.to(DEVICE, non_blocking=True) for b in batch
                        ]
                        T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                        
                        T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                        
                        seq_len = P0.shape[1]
                        mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                        
                        # MSE
                        mse_0 = mse_criterion(T0_pred, T0_true)
                        mse_1 = mse_criterion(T1_pred, T1_true)
                        masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                        
                        # MAE
                        mae_0 = mae_criterion(T0_pred, T0_true)
                        mae_1 = mae_criterion(T1_pred, T1_true)
                        masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())
                        
                        val_mse_accum += masked_mse.item()
                        val_mae_accum += masked_mae.item()
                        val_steps += 1
                        
                    del chunk_data, dataset, dataloader
                    
            avg_val_mse = val_mse_accum / val_steps
            val_rmse = avg_val_mse ** 0.5
            val_mae = val_mae_accum / val_steps
            
            epoch_time = time.perf_counter() - epoch_start_time
            
            # 3. PRINT PROGRESS
            p = {k: v.item() for k, v in model.get_physical_params().items()}
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Time: {epoch_time:.1f}s | Train RMSE: {train_rmse:.4f} °C (MAE: {train_mae:.4f}) | Val RMSE: {val_rmse:.4f} °C (MAE: {val_mae:.4f}) | LR: {current_lr:.6f}")
            print(f"Params: C: {p['C']:.2f} | h_base0: {p['h_base_0']:.2f} | h_act0: {p['h_active_0']:.2f} | T_thr0: {p['T_thresh_0']:.1f} | h_base1: {p['h_base_1']:.2f} | h_act1: {p['h_active_1']:.2f} | T_thr1: {p['T_thresh_1']:.1f}")

            scheduler.step(val_rmse)
            
            # 4. LOGGING TO CSV
            with open(CSV_LOG_PATH, "a") as f:
                f.write(f"{epoch},{train_rmse:.6f},{val_rmse:.6f},{train_mae:.6f},{val_mae:.6f},{current_lr:.6f},{p['C']:.4f},{p['h_base_0']:.4f},{p['h_base_1']:.4f},{p['h_active_0']:.4f},{p['h_active_1']:.4f},{p['T_thresh_0']:.4f},{p['T_thresh_1']:.4f},{p['beta_0']:.4f},{p['beta_1']:.4f},{p['k01']:.4f},{p['k10']:.4f},{p['q0']:.4f},{p['q1']:.4f},{epoch_time:.2f}\n")
    
            # 5. CHECKPOINTING
            is_best = val_rmse < best_val_loss
            if is_best:
                best_val_loss = val_rmse  # Update best_val_loss before saving
    
            checkpoint_state = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'torch_rng': torch.get_rng_state(),
                'numpy_rng': np.random.get_state(),
                'cuda_rng': torch.cuda.get_rng_state() if torch.cuda.is_available() else None
            }
    
            # Save standard resume checkpoint (now contains the updated best_val_loss)
            torch.save(checkpoint_state, RESUME_PATH)
    
            # Save best model checkpoint if validation loss improves
            if is_best:
                torch.save(checkpoint_state, BEST_PATH)
                
                with open(SAVE_DIR / "calibrated_physics.json", "w") as f:
                    json.dump(p, f, indent=4)
                print(">>> New best parameters saved! <<<")
    
        # --- END SUMMARY ---
        end_time = time.perf_counter()
        formatted_time = str(timedelta(seconds=int(end_time - start_time)))
        
        print("\n" + "=" * 70)
        print("=== CALIBRATION COMPLETE ===")
        print(f"Total Time : {formatted_time}")
        print(f"Best Val RMSE : {best_val_loss:.4f} °C")
        print(f"Results saved to: {SAVE_DIR}")
        print("=" * 70)

    finally:
        # This guarantees the background GPU logger is killed even if the script crashes
        if gpu_logger_process is not None:
            gpu_logger_process.terminate()
            gpu_logger_process.wait()
            print("[*] Background GPU logger terminated.")

        # Safely detach the dual logger
        if hasattr(sys.stdout, 'terminal'):
            sys.stdout = sys.stdout.terminal

if __name__ == "__main__":
    main()


[SYSTEM] Live GPU Status:
Thu Apr  2 15:53:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 30%   48C    P8             18W /  200W |    7870MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.



Enter version prefix (e.g., v1_base) [Press Enter for default 'v1']:  ode_v2


--- STARTING: 07_GPU_ODE_CALIBRATION ---
[*] Version Prefix: ode_v2
[*] Device: cuda:1
[*] Saving to: Capstone/Implementation/models/ode_v2
[*] Started continuous GPU logging to: ode_v2_gpu1_timeseries.csv

[SYSTEM] No checkpoint found. Starting fresh training.
Found 7 train chunks and 1 val chunks.

--- Epoch 1/50 ---


...

Time: 409.0s | Train RMSE: 4.5652 °C (MAE: 3.3977) | Val RMSE: 4.6707 °C (MAE: 3.6988) | LR: 0.010000
Params: C: 209.31 | h_base0: 3.76 | h_act0: 4.73 | T_thr0: 69.2 | h_base1: 3.89 | h_act1: 6.64 | T_thr1: 64.6
>>> New best parameters saved! <<<

--- Epoch 18/50 ---


Time: 414.8s | Train RMSE: 4.5348 °C (MAE: 3.3668) | Val RMSE: 4.6436 °C (MAE: 3.6670) | LR: 0.010000
Params: C: 211.92 | h_base0: 3.78 | h_act0: 4.79 | T_thr0: 69.5 | h_base1: 3.94 | h_act1: 6.56 | T_thr1: 64.9
>>> New best parameters saved! <<<

--- Epoch 19/50 ---


Time: 412.1s | Train RMSE: 4.5092 °C (MAE: 3.3406) | Val RMSE: 4.6211 °C (MAE: 3.6395) | LR: 0.010000
Params: C: 214.51 | h_base0: 3.79 | h_act0: 4.86 | T_thr0: 69.7 | h_base1: 3.97 | h_act1: 6.51 | T_thr1: 65.1
>>> New best parameters saved! <<<

--- Epoch 20/50 ---


Time: 415.2s | Train RMSE: 4.4889 °C (MAE: 3.3189) | Val RMSE: 4.6021 °C (MAE: 3.6154) | LR: 0.010000
Params: C: 217.06 | h_base0: 3.80 | h_act0: 4.95 | T_thr0: 69.9 | h_base1: 4.01 | h_act1: 6.48 | T_thr1: 65.4
>>> New best parameters saved! <<<

--- Epoch 21/50 ---


Train Chunk 6/7:   0%|          | 0/2 [00:00<?, ?it/s]        

In [ ]:
"""
===============================================================================
Script Name: 01_gpu_ode_calibration.py
Description: Loads chunked tensor segments and trains a Physics-Informed 
             ODE to calibrate thermal parameters. Implements a dynamic
             state-dependent cooling proxy h(T) to simulate fan curves.
             Includes interactive GPU selection, auto-resume, and CSV logging.
===============================================================================
"""

import os
import sys
import time
import warnings
import json
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from tqdm import tqdm
from datetime import timedelta
import gc

# --- 1. CONFIGURATION ---
warnings.filterwarnings('ignore')

# Hyperparameters
BATCH_SIZE = 25000
EPOCHS = 50
LEARNING_RATE = 0.01
DT = 0.11  # Timestep in seconds

# ==========================================
# 2. GPU SELECTION & INITIALIZATION
# ==========================================
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt (in that order).
    """
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            if gpu_id < torch.cuda.device_count():
                print(f"[SYSTEM] GPU {gpu_id} selected via CLI argument.")
                return gpu_id
            else:
                print(f"[!] CLI GPU {gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        gpu_id = int(env_gpu)
        if gpu_id < torch.cuda.device_count():
            print(f"[SYSTEM] GPU {gpu_id} selected via GPU_ID environment variable.")
            return gpu_id
        else:
            print(f"[!] Env GPU_ID={gpu_id} not found. Available: 0-{torch.cuda.device_count()-1}")

    print("\n[SYSTEM] Live GPU Status:")
    try:
        subprocess.run(["nvidia-smi"])
    except FileNotFoundError:
        print("[WARNING] nvidia-smi not found. Cannot display GPU stats.")

    if not torch.cuda.is_available():
        print("[WARNING] No CUDA GPUs detected. Falling back to CPU.")
        return None

    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']: ").strip()
        if not gpu_id:
            print("[SYSTEM] Defaulting to GPU 0.")
            return 0
        if gpu_id.isdigit():
            gpu_id_int = int(gpu_id)
            if gpu_id_int < torch.cuda.device_count():
                print(f"[SYSTEM] Manually locking script to GPU {gpu_id_int}.")
                return gpu_id_int
            else:
                print(f"[!] GPU {gpu_id_int} not found. Available GPUs: 0-{torch.cuda.device_count()-1}")
                continue
        print("[!] Invalid input. Please enter a valid integer ID.")

# --- 3. TERMINAL LOGGING UTILITY ---
class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "a", encoding="utf-8") # Append mode for resuming

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# --- 4. PHYSICS MODEL (THE ODE) ---
class ThermalODEModel(nn.Module):
    def __init__(self):
        super(ThermalODEModel, self).__init__()
        
        # 1. Define Priors (Starting Guesses)
        prior_C         = 141.735
        prior_k01       = 0.0780
        prior_k10       = 0.0281
        prior_q         = 0.0
        
        # Dynamic Fan Curve Priors
        prior_h_base    = 2.0   # Baseline passive cooling
        prior_h_active  = 8.0   # Max additional cooling from fans
        prior_T_thresh  = 65.0  # Temp where fans start to aggressively ramp up
        prior_beta      = 0.5   # Steepness of the fan curve ramp
        
        # 2. Define Physical Bounds [lower, upper]
        self.bounds = {
            'C':          (50.0, 300.0),
            'k01':        (0.0, 0.5),
            'k10':        (0.0, 0.5),
            'q0':         (-10.0, 10.0),
            'q1':         (-10.0, 10.0),
            'h_base_0':   (0.5, 10.0),
            'h_base_1':   (0.5, 10.0),
            'h_active_0': (0.0, 30.0),
            'h_active_1': (0.0, 30.0),
            'T_thresh_0': (40.0, 85.0),
            'T_thresh_1': (40.0, 85.0),
            'beta_0':     (0.05, 2.0),
            'beta_1':     (0.05, 2.0)
        }
        
        # 3. Initialize Raw Parameters (Inverse Sigmoid)
        def inv_sigmoid(val, low, high):
            norm = (val - low) / (high - low)
            norm = max(min(norm, 0.999), 0.001)
            return torch.log(torch.tensor(norm / (1.0 - norm)))

        # General Params
        self.raw_C          = nn.Parameter(inv_sigmoid(prior_C, *self.bounds['C']))
        self.raw_k01        = nn.Parameter(inv_sigmoid(prior_k01, *self.bounds['k01']))
        self.raw_k10        = nn.Parameter(inv_sigmoid(prior_k10, *self.bounds['k10']))
        self.raw_q0         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q0']))
        self.raw_q1         = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q1']))
        
        # GPU 0 Fan Curve Params
        self.raw_h_base_0   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_0']))
        self.raw_h_active_0 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_0']))
        self.raw_T_thresh_0 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_0']))
        self.raw_beta_0     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_0']))
        
        # GPU 1 Fan Curve Params
        self.raw_h_base_1   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_1']))
        self.raw_h_active_1 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_1']))
        self.raw_T_thresh_1 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_1']))
        self.raw_beta_1     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_1']))

    def get_physical_params(self):
        """Applies sigmoid to raw parameters to bind them within physical reality."""
        p = {}
        for name, bound in self.bounds.items():
            raw_val = getattr(self, f"raw_{name}")
            low, high = bound
            p[name] = low + torch.sigmoid(raw_val) * (high - low)
        return p

    def forward(self, P0, P1, T0_init, T1_init, T_amb):
        """Simulates the thermal trajectories using Forward Euler integration."""
        params = self.get_physical_params()
        
        batch_size, seq_len = P0.shape
        
        # Pre-allocate to prevent torch.stack memory fragmentation
        T0_preds = torch.empty((batch_size, seq_len), device=P0.device)
        T1_preds = torch.empty((batch_size, seq_len), device=P1.device)
        
        T0_curr = T0_init
        T1_curr = T1_init
        T0_preds[:, 0] = T0_curr
        T1_preds[:, 0] = T1_curr
        
        inv_C = 1.0 / params['C']
        
        # Euler Integration Loop
        for t in range(seq_len - 1):
            # 1. Calculate Dynamic Cooling Rate h(T) for current timestep
            h0_curr = params['h_base_0'] + params['h_active_0'] * torch.sigmoid(params['beta_0'] * (T0_curr - params['T_thresh_0']))
            h1_curr = params['h_base_1'] + params['h_active_1'] * torch.sigmoid(params['beta_1'] * (T1_curr - params['T_thresh_1']))
            
            # 2. Calculate differentials
            dT0 = (P0[:, t] + params['k01'] * P1[:, t] - h0_curr * T0_curr + h0_curr * T_amb + params['q0']) * inv_C
            dT1 = (P1[:, t] + params['k10'] * P0[:, t] - h1_curr * T1_curr + h1_curr * T_amb + params['q1']) * inv_C
            
            # 3. Euler Step
            T0_curr = T0_curr + DT * dT0
            T1_curr = T1_curr + DT * dT1
            
            T0_preds[:, t+1] = T0_curr
            T1_preds[:, t+1] = T1_curr
            
        return T0_preds, T1_preds


# --- 5. MAIN EXECUTION ---
def main():
    global script_name
    try:
        # Works when running as a normal .py script
        script_path = Path(__file__).resolve()
        script_name = script_path.stem
        project_root = script_path.parent.parent.parent
    except NameError:
        # Fallback for Jupyter Notebooks
        script_name = "01_gpu_ode_calibration"
        current_dir = Path.cwd() 
        project_root = current_dir.parent.parent
    
    # --- Device Setup ---
    gpu_id = select_gpu()
    if gpu_id is not None:
        torch.cuda.set_device(gpu_id)
        DEVICE = torch.device(f"cuda:{gpu_id}")
    else:
        DEVICE = torch.device("cpu")

    # --- Versioning & Path Initialization ---
    user_prefix = input("\nEnter version prefix (e.g., v1_base) [Press Enter for default 'v1']: ").strip()
    VERSION_PREFIX = user_prefix if user_prefix else "v1"

    # Input Data Directory
    data_dir = project_root / "Implementation" / "data" / "mit-supercloud-dataset" / "gpu" / "dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_tensors"
    if not data_dir.exists():
        print(f"[ERROR] Cannot find tensor directory: {data_dir}")
        return

    prefix = "/home/sanke24839/"
    # Output Save Directory
    SAVE_DIR = project_root / "Implementation" / "models" / VERSION_PREFIX
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    
    log_path = SAVE_DIR / f"{script_name}_terminal_output.txt"
    sys.stdout = DualLogger(log_path)
    
    start_time = time.perf_counter()
    print("=" * 70)
    print(f"--- STARTING: {script_name.upper()} ---")
    print(f"[*] Version Prefix: {VERSION_PREFIX}")
    print(f"[*] Device: {DEVICE}")
    print(f"[*] Saving to: {str(SAVE_DIR).replace(prefix, '')}")
    print("=" * 70)

    # --- Background GPU Time-Series Logging ---
    gpu_logger_process = None
    if gpu_id is not None:
        gpu_csv_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu{gpu_id}_timeseries.csv"
        try:
            # Queries specific metrics for the chosen GPU (-i), loops every 5 seconds (-l 5), writes to file (-f)
            smi_cmd = [
                "nvidia-smi",
                "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu,power.draw",
                "--format=csv",
                "-i", str(gpu_id),
                "-l", "5", 
                "-f", str(gpu_csv_path)
            ]
            gpu_logger_process = subprocess.Popen(smi_cmd)
            print(f"[*] Started continuous GPU logging to: {gpu_csv_path.name}")
        except FileNotFoundError:
            print("[WARNING] nvidia-smi not found. Continuous GPU logging disabled.")

    # --- Setup Model, Optimizer, & Criterions ---
    model = ThermalODEModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    mse_criterion = nn.MSELoss(reduction='none') # For optimization
    mae_criterion = nn.L1Loss(reduction='none')  # For interpretable logging

    # --- Auto-Resume Logic ---
    RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
    BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
    CSV_LOG_PATH = SAVE_DIR / "training_log.csv"

    start_epoch = 1
    best_val_loss = float('inf')

    if RESUME_PATH.exists():
        print(f"\n[SYSTEM] Found existing checkpoint at: {RESUME_PATH.name}")
        print(f"[SYSTEM] Resuming training...")
        
        checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        
        # Restore RNG states
        torch.set_rng_state(checkpoint['torch_rng'].cpu().to(torch.uint8) if isinstance(checkpoint['torch_rng'], torch.Tensor) else torch.ByteTensor(checkpoint['torch_rng']))
        np.random.set_state(checkpoint['numpy_rng'])
        
        if torch.cuda.is_available() and 'cuda_rng' in checkpoint and checkpoint['cuda_rng'] is not None:
            cuda_rng = checkpoint['cuda_rng']
            torch.cuda.set_rng_state(cuda_rng.cpu().to(torch.uint8) if isinstance(cuda_rng, torch.Tensor) else torch.ByteTensor(cuda_rng))
            
        print(f"[SYSTEM] Resumed from Epoch {checkpoint['epoch']}. Best Val RMSE: {best_val_loss:.6f}")
    else:
        print(f"\n[SYSTEM] No checkpoint found. Starting fresh training.")
        # Initialize CSV header if starting fresh
        with open(CSV_LOG_PATH, "w") as f:
            f.write("epoch,train_rmse,val_rmse,train_mae,val_mae,current_lr,C,h_base_0,h_base_1,h_active_0,h_active_1,T_thresh_0,T_thresh_1,beta_0,beta_1,k01,k10,q0,q1,epoch_time_sec\n")

    # Discover chunks
    train_chunks = sorted(list(data_dir.glob("train_segments_part*.pt")))
    val_chunks = sorted(list(data_dir.glob("val_segments_part*.pt")))
    print(f"Found {len(train_chunks)} train chunks and {len(val_chunks)} val chunks.")

    try:
        # --- Training Loop ---
        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_start_time = time.perf_counter()
            print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
            
            # 1. TRAINING PHASE
            model.train()
            train_mse_accum = 0.0
            train_mae_accum = 0.0
            train_steps = 0
            
            for chunk_idx, chunk_file in enumerate(train_chunks):
                chunk_data = torch.load(chunk_file, map_location='cpu', weights_only=False)
                dataset = TensorDataset(
                    chunk_data['P0'], chunk_data['P1'], 
                    chunk_data['T0'], chunk_data['T1'],
                    chunk_data['T_amb'], chunk_data['valid_len']
                )
                dataloader = DataLoader(
                    dataset,
                    batch_size=BATCH_SIZE,
                    shuffle=True,
                    num_workers=4,          # parallel CPU workers prefetch data
                    pin_memory=True,        # faster CPU→GPU transfer
                    persistent_workers=True
                )
                    
                for batch in tqdm(dataloader, desc=f"Train Chunk {chunk_idx+1}/{len(train_chunks)}", leave=False):
                    P0, P1, T0_true, T1_true, T_amb, valid_len = [
                        b.to(DEVICE, non_blocking=True) for b in batch
                    ]
                    
                    optimizer.zero_grad()
                    
                    # Forward pass
                    T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                    T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                    
                    # Create mask for valid timesteps
                    seq_len = P0.shape[1]
                    mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                    
                    # Calculate MSE (for backprop)
                    mse_0 = mse_criterion(T0_pred, T0_true)
                    mse_1 = mse_criterion(T1_pred, T1_true)
                    masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                    
                    # Calculate MAE (for tracking interpretability)
                    with torch.no_grad():
                        mae_0 = mae_criterion(T0_pred.detach(), T0_true)
                        mae_1 = mae_criterion(T1_pred.detach(), T1_true)
                        masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())
    
                    # Backpropagate
                    masked_mse.backward()
                    optimizer.step()
                    
                    train_mse_accum += masked_mse.item()
                    train_mae_accum += masked_mae.item()
                    train_steps += 1
                    
                del chunk_data, dataset, dataloader
    
            avg_train_mse = train_mse_accum / train_steps
            train_rmse = avg_train_mse ** 0.5
            train_mae = train_mae_accum / train_steps
            
            # 2. VALIDATION PHASE
            model.eval()
            val_mse_accum = 0.0
            val_mae_accum = 0.0
            val_steps = 0
            
            with torch.no_grad():
                for chunk_idx, chunk_file in enumerate(val_chunks):
                    chunk_data = torch.load(chunk_file, map_location='cpu', weights_only=False)
                    dataset = TensorDataset(
                        chunk_data['P0'], chunk_data['P1'], 
                        chunk_data['T0'], chunk_data['T1'],
                        chunk_data['T_amb'], chunk_data['valid_len']
                    )
                    dataloader = DataLoader(
                        dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        num_workers=4,          # parallel CPU workers prefetch data
                        pin_memory=True,        # faster CPU→GPU transfer
                        persistent_workers=True
                    )
                    
                    for batch in dataloader:
                        P0, P1, T0_true, T1_true, T_amb, valid_len = [
                            b.to(DEVICE, non_blocking=True) for b in batch
                        ]
                        T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                        
                        T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                        
                        seq_len = P0.shape[1]
                        mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                        
                        # MSE
                        mse_0 = mse_criterion(T0_pred, T0_true)
                        mse_1 = mse_criterion(T1_pred, T1_true)
                        masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                        
                        # MAE
                        mae_0 = mae_criterion(T0_pred, T0_true)
                        mae_1 = mae_criterion(T1_pred, T1_true)
                        masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())
                        
                        val_mse_accum += masked_mse.item()
                        val_mae_accum += masked_mae.item()
                        val_steps += 1
                        
                    del chunk_data, dataset, dataloader
                    
            avg_val_mse = val_mse_accum / val_steps
            val_rmse = avg_val_mse ** 0.5
            val_mae = val_mae_accum / val_steps
            
            epoch_time = time.perf_counter() - epoch_start_time
            
            # 3. PRINT PROGRESS
            p = {k: v.item() for k, v in model.get_physical_params().items()}
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Time: {epoch_time:.1f}s | Train RMSE: {train_rmse:.4f} °C (MAE: {train_mae:.4f}) | Val RMSE: {val_rmse:.4f} °C (MAE: {val_mae:.4f}) | LR: {current_lr:.6f}")
            print(f"Params: C: {p['C']:.2f} | h_base0: {p['h_base_0']:.2f} | h_act0: {p['h_active_0']:.2f} | T_thr0: {p['T_thresh_0']:.1f} | h_base1: {p['h_base_1']:.2f} | h_act1: {p['h_active_1']:.2f} | T_thr1: {p['T_thresh_1']:.1f}")

            scheduler.step(val_rmse)
            
            # 4. LOGGING TO CSV
            with open(CSV_LOG_PATH, "a") as f:
                f.write(f"{epoch},{train_rmse:.6f},{val_rmse:.6f},{train_mae:.6f},{val_mae:.6f},{current_lr:.6f},{p['C']:.4f},{p['h_base_0']:.4f},{p['h_base_1']:.4f},{p['h_active_0']:.4f},{p['h_active_1']:.4f},{p['T_thresh_0']:.4f},{p['T_thresh_1']:.4f},{p['beta_0']:.4f},{p['beta_1']:.4f},{p['k01']:.4f},{p['k10']:.4f},{p['q0']:.4f},{p['q1']:.4f},{epoch_time:.2f}\n")
    
            # 5. CHECKPOINTING
            is_best = val_rmse < best_val_loss
            if is_best:
                best_val_loss = val_rmse  # Update best_val_loss before saving
    
            checkpoint_state = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'torch_rng': torch.get_rng_state(),
                'numpy_rng': np.random.get_state(),
                'cuda_rng': torch.cuda.get_rng_state() if torch.cuda.is_available() else None
            }
    
            # Save standard resume checkpoint (now contains the updated best_val_loss)
            torch.save(checkpoint_state, RESUME_PATH)
    
            # Save best model checkpoint if validation loss improves
            if is_best:
                torch.save(checkpoint_state, BEST_PATH)
                
                with open(SAVE_DIR / "calibrated_physics.json", "w") as f:
                    json.dump(p, f, indent=4)
                print(">>> New best parameters saved! <<<")
    
        # --- END SUMMARY ---
        end_time = time.perf_counter()
        formatted_time = str(timedelta(seconds=int(end_time - start_time)))
        
        print("\n" + "=" * 70)
        print("=== CALIBRATION COMPLETE ===")
        print(f"Total Time : {formatted_time}")
        print(f"Best Val RMSE : {best_val_loss:.4f} °C")
        print(f"Results saved to: {SAVE_DIR}")
        print("=" * 70)

    finally:
        # This guarantees the background GPU logger is killed even if the script crashes
        if gpu_logger_process is not None:
            gpu_logger_process.terminate()
            gpu_logger_process.wait()
            print("[*] Background GPU logger terminated.")

        # Safely detach the dual logger
        if hasattr(sys.stdout, 'terminal'):
            sys.stdout = sys.stdout.terminal

if __name__ == "__main__":
    main()


[SYSTEM] Live GPU Status:
Thu Apr  2 18:42:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 36%   67C    P2             85W /  200W |    7870MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter the GPU ID you want to allocate (e.g., 0, 1) [Press Enter for '0']:  1


[SYSTEM] Manually locking script to GPU 1.



Enter version prefix (e.g., v1_base) [Press Enter for default 'v1']:  ode_v2


--- STARTING: 07_GPU_ODE_CALIBRATION ---
[*] Version Prefix: ode_v2
[*] Device: cuda:1
[*] Saving to: Capstone/Implementation/models/ode_v2
[*] Started continuous GPU logging to: ode_v2_gpu1_timeseries.csv

[SYSTEM] Found existing checkpoint at: ode_v2_resume_checkpoint.pt
[SYSTEM] Resuming training...
[SYSTEM] Resumed from Epoch 20. Best Val RMSE: 4.602055
Found 7 train chunks and 1 val chunks.

--- Epoch 21/50 ---


Time: 419.8s | Train RMSE: 4.4716 °C (MAE: 3.3005) | Val RMSE: 4.5857 °C (MAE: 3.5941) | LR: 0.010000
Params: C: 219.58 | h_base0: 3.81 | h_act0: 5.04 | T_thr0: 70.1 | h_base1: 4.04 | h_act1: 6.47 | T_thr1: 65.6
>>> New best parameters saved! <<<

--- Epoch 22/50 ---


Train Chunk 1/7:   0%|          | 0/2 [00:00<?, ?it/s]